# 1110 48일차

## 1. 동시 요청

요청을 한꺼번에 보내 동시 요청 수에 따라 처리량이 어떻게 바뀌는지 보는 것 (`f21_continuous_batching1`, `f22_continuous_batching2`)

요청을 하나씩 보내면 한 번에 토큰 하나씩만 만들어서 장치의 연산 여력이 남음. 서버는 들어온 요청을 묶어 같은 단계에서 함께 처리해서(continuous batching) 동시에 보낼수록 전체 시간이 줄어듦

1. `prompts = [f"... {i}번 ..." for i in range(8)]` : 질문 8개를 리스트로 만듦
2. `ThreadPoolExecutor(max_workers=workers)` : 동시에 보낼 요청 수를 정함
3. `pool.map(ask, prompts)` : 질문마다 함수를 호출하고 결과를 순서대로 모음
4. `for workers in [1, 2, 4, 8]:` : 동시 요청 수만 바꿔 같은 측정을 반복함
5. `statistics.mean(...)` : 요청별 지연 시간의 평균

- 동시 요청 수를 늘리면 전체 시간이 줄고 TPS가 올라감
- **요청 하나하나의 지연은 조금씩 늘어남. 처리량과 지연이 같이 좋아지지는 않음**
- 전체 출력 토큰 수가 같아야 견줄 수 있어서 `max_tokens`를 고정해 둠
- `with` 블록을 벗어날 때 보낸 요청이 모두 끝나길 기다림
- `ThreadPoolExecutor`는 요청을 보내고 기다리는 일만 나눠 맡음. 토큰을 만드는 쪽은 서버임

## 2. 측정 함수

호출 한 번에 걸린 시간과 토큰 수, 본문을 한 함수에서 딕셔너리로 돌려주는 것 (`common.py`의 `run_once`)

스크립트마다 측정 코드를 다시 적으면 재는 자리가 조금씩 달라져서 결과를 견줄 수 없음. 한 함수에 모아 같은 방식으로 재게 함

1. `def run_once(prompt, max_tokens=1000):` : 질문과 출력 상한을 받아 한 번 호출함
2. `usage = response.usage_metadata or {}` : 사용량이 비어 있으면 빈 딕셔너리로 둠
3. `usage.get('input_tokens', 0)` : 키가 없으면 0을 돌려줌
4. `return {"elapsed": …, "output_tokens": …, "text": …}` : 필요한 값만 담아 돌려줌

- 딕셔너리로 돌려주면 받는 쪽에서 `r["elapsed"]`처럼 이름으로 꺼냄
- **`usage['input_tokens']`는 키가 없으면 에러가 남. `.get(키, 0)`은 기본값을 돌려줘서 측정이 멈추지 않음**
- 측정에 쓰는 `perf_counter`도 함수 안에 들어가서 호출마다 같은 구간을 잼

## 3. KV Cache

서버가 입력을 처리하며 만들어 둔 중간 값을 들고 있다가 재사용하는 공간 (`f22_kv_cache1`, `f22_kv_cache2`)

입력이 길어질수록 계산할 양이 늘어남. 앞부분 계산 결과가 남아 있으면 같은 앞부분이 다시 들어올 때 그만큼 건너뜀

1. `"NPU는 인공지능 연산을 가속하는 프로세서다." * 100` : 같은 문장을 반복해 긴 입력을 만듦
2. `tests = [("짧은 입력", short_prompt, 80), …]` : 이름·질문·출력 상한을 튜플로 묶어 차례로 돌림
3. `http://127.0.0.1:8000/metrics` : 서버가 상태 수치를 내주는 주소
4. `urllib.request.urlopen(url, timeout=5)` : 그 주소를 열어 본문을 읽음
5. `line.split()[-1]` : 한 줄의 마지막 칸이 값이라 그것만 꺼내 더함

- `furiosa_llm_prefix_cache_queries`는 조회 횟수, `_hits`는 그중 재사용한 횟수. 둘을 나누면 적중률
- `furiosa_llm_kv_cache_usage_percent`는 비율로 와서 100을 곱해 퍼센트로 봄
- `#`로 시작하는 줄은 설명이라 건너뜀
- **쌓이기만 하는 수치라서 한 번 읽은 값이 아니라 호출 전후의 차이를 봐야 함**
- 입력 토큰이 몇 배로 늘어도 응답 시간은 거의 같음. 출력 토큰 수를 같게 두었기 때문
- 같은 문장을 반복한 입력은 앞부분이 겹쳐서 적중률이 올라감
- `input()`으로 중간에 멈춰 두고 서버 쪽 사용률을 직접 보기도 함

#### cf) lambda로 인자 고정

- `pool.map`은 리스트의 값 하나만 함수에 넘겨서, 다른 인자까지 주려면 함수를 감싸야 함
- `def ask(prompt): return run_once(prompt, 100)`과 `lambda p: run_once(p, 50)`은 같은 일을 함
- 한 번만 쓸 함수면 `lambda`로 그 자리에 적음